# 🔄 Step 2: Preprocessing & NDVI Calculation (2024)

Calculate NDVI and NDWI for 2024 satellite data.

## What This Notebook Does:
- ✅ Load bands from 2024 data  
- ✅ Calculate NDVI (Normalized Difference Vegetation Index)
- ✅ Calculate NDWI (Normalized Difference Water Index)
- ✅ Compute texture features
- ✅ Stack all indices into processed image
- ✅ **Skip Ground Truth** - using trained 2025 model for prediction only

---

**Previous:** [01_data_loading_2024.ipynb](01_data_loading_2024.ipynb)  
**Next:** [04_prediction_analysis_2024.ipynb](04_prediction_analysis_2024.ipynb)  
**Note:** We skip 03 (model training) because we use the existing 2025 trained model

## Imports

In [ ]:
import rasterio
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")

## Load 2024 Bands from Pickle

In [ ]:
import pickle

# Load bands saved from previous notebook
print("📂 Loading 2024 bands...")

with open('outputs/bands_2024.pkl', 'rb') as f:
    bands_data = pickle.load(f)

blue = bands_data['B02']
green = bands_data['B03']
red = bands_data['B04']
nir = bands_data['B08']

print(f"✅ Bands loaded!")
print(f"   Shape: {blue.shape}")
print(f"   Total pixels: {blue.size:,}")

## Calculate Vegetation Indices

**NDVI Formula:** `(NIR - Red) / (NIR + Red)`  
**NDWI Formula:** `(Green - NIR) / (Green + NIR)`  
**Texture:** Local variance in NIR band

In [ ]:
print("📊 Calculating vegetation indices...")
np.seterr(divide='ignore', invalid='ignore')

# 1. NDVI (Vegetation Index)
ndvi = (nir - red) / (nir + red + 1e-10)

# 2. NDWI (Water Index)  
ndwi = (green - nir) / (green + nir + 1e-10)

# 3. TEXTURE (Variance/Roughness)
print("   Calculating texture (3x3 local variance)...")
mean = uniform_filter(nir, size=3)
sqr_mean = uniform_filter(nir**2, size=3)
var = sqr_mean - mean**2
texture = np.sqrt(np.maximum(var, 0))

# Clean up NaNs
ndvi = np.nan_to_num(ndvi, nan=0)
ndwi = np.nan_to_num(ndwi, nan=0)
texture = np.nan_to_num(texture, nan=0)

print(f"✅ Indices calculated!")
print(f"   NDVI range: [{ndvi.min():.4f}, {ndvi.max():.4f}]")
print(f"   NDWI range: [{ndwi.min():.4f}, {ndwi.max():.4f}]")
print(f"   Texture range: [{texture.min():.4f}, {texture.max():.4f}]")

## Visualize Indices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# NDVI
im = axes[0, 0].imshow(ndvi, cmap='RdYlGn', vmin=-1, vmax=1)
axes[0, 0].set_title('NDVI (2024)\nGreen=Vegetation, Red=Water', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=axes[0, 0], label='NDVI')
axes[0, 0].set_xlabel('Pixel Column')
axes[0, 0].set_ylabel('Pixel Row')

# NDWI
im = axes[0, 1].imshow(ndwi, cmap='BrBG', vmin=-1, vmax=1)
axes[0, 1].set_title('NDWI (2024)\nGreen=Water, Brown=Land', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=axes[0, 1], label='NDWI')
axes[0, 1].set_xlabel('Pixel Column')
axes[0, 1].set_ylabel('Pixel Row')

# Texture
im = axes[1, 0].imshow(texture, cmap='gray')
axes[1, 0].set_title('Texture (Local Variance) - 2024', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=axes[1, 0], label='Variance')
axes[1, 0].set_xlabel('Pixel Column')
axes[1, 0].set_ylabel('Pixel Row')

# RGB Composite (B04, B03, B02 = Red, Green, Blue)
# Normalize to 0-255 for display
rgb = np.stack([red, green, blue], axis=2)
rgb_norm = (rgb - rgb.min()) / (rgb.max() - rgb.min()) * 255
axes[1, 1].imshow(rgb_norm.astype(np.uint8))
axes[1, 1].set_title('RGB Composite (B04, B03, B02) - 2024', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Pixel Column')
axes[1, 1].set_ylabel('Pixel Row')

plt.tight_layout()
plt.savefig('outputs/02_indices_2024.png', dpi=300, bbox_inches='tight')
plt.show()

print("💾 Visualization saved to 'outputs/02_indices_2024.png'")

## Stack All Bands into 7-Layer GeoTIFF

In [ ]:
# Get CRS and transform from original data
print("📦 Stacking all bands...")

# Read metadata from original 2024 data
data_folder_2024 = "coastalImage_2024"
all_files = os.listdir(data_folder_2024)
first_band = None

for f in all_files:
    if 'B02' in f and (f.endswith('.tif') or f.endswith('.tiff')):
        first_band = os.path.join(data_folder_2024, f)
        break

if first_band:
    with rasterio.open(first_band) as src:
        profile = src.profile.copy()
else:
    # Fallback profile
    profile = {
        'driver': 'GTiff',
        'dtype': 'float32',
        'crs': 'EPSG:4326',
        'transform': rasterio.transform.from_bounds(0, 0, 1, 1, blue.shape[1], blue.shape[0])
    }

# Stack all 7 layers: B02, B03, B04, B08, NDVI, NDWI, Texture
stacked_image = np.stack([
    blue, green, red, nir,
    ndvi, ndwi, texture
]).astype('float32')

# Update metadata
profile.update(
    count=7,
    dtype='float32',
    driver='GTiff'
)

# Save as GeoTIFF
output_path = "processed_image_with_indices_2024.tif"
with rasterio.open(output_path, 'w', **profile) as dst:
    dst.write(stacked_image)

print(f"✅ Stacked image saved!")
print(f"   Output: {output_path}")
print(f"   Shape: {stacked_image.shape} (7 bands, {stacked_image.shape[1]}x{stacked_image.shape[2]} pixels)")
print(f"\n📋 Band Layout:")
print(f"   1. B02 (Blue)")
print(f"   2. B03 (Green)")
print(f"   3. B04 (Red)")
print(f"   4. B08 (NIR)")
print(f"   5. NDVI (Vegetation Index)")
print(f"   6. NDWI (Water Index)")
print(f"   7. Texture (Local Variance)")

## Summary

In [ ]:
print("\n" + "="*70)
print("  ✅ 2024 DATA PREPROCESSING COMPLETE")
print("="*70)
print(f"\n📊 Created: processed_image_with_indices_2024.tif")
print(f"\n📝 Key Differences from 2025 Training Data:")
print(f"   • NEW satellite imagery from 2024")
print(f"   • Using EXISTING trained 2025 model for predictions")
print(f"   • NO Ground Truth data needed")
print(f"\n🔮 Next Step: Open 04_prediction_analysis_2024.ipynb")
print(f"    This will apply the 2025-trained model to 2024 data")
print("="*70)